## Upload QA pairs to Hugging Face Datasets


**What gets uploaded**:
Question-answer pair files under `data/question_answer_pairs/<category>_qa_pairs_<workload_type>.jsonl` (e.g. `cs.AI_qa_pairs_short_factual.jsonl`), uploaded to `question_answer_pairs/<workload_type>/<category>.jsonl` on the Hub

**Before running**: you need a Hugging Face account and a private dataset repo already created (or this notebook will create one for you on first run, see the config cell).

### Install dependencies (Colab only, skip if running locally with them already installed)

In [1]:
# Uncomment if running in Colab or a fresh environment
!pip install huggingface_hub -q


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Configuration


In [2]:
import os

# --- Repo settings ---
REPO_ID = "Sakhiur/empirical-rag-paradigm-benchmark"  
REPO_TYPE = "dataset"
PRIVATE = False 

# --- Local paths, relative to this notebook's location under notebooks/ ---
PROJECT_ROOT = ".."  # adjust if this notebook moves
QA_PAIRS_DIR = os.path.join(PROJECT_ROOT, "data", "question_answer_pairs")

# The 10 target categories, used only to validate what we find, not to require all of them
ALL_CATEGORIES = [
    "cs.AI", "cs.LG", "cs.IR", "cs.DB", "cs.SE",       # yours
    "cs.CV", "cs.CL", "cs.NE", "cs.DC", "cs.CR",       # collaborator's
]

# Workload types from the proposal's taxonomy (W1/W2/W3), matched against filenames
# shaped like '<category>_qa_pairs_<workload_type>.jsonl', e.g. 'cs.AI_qa_pairs_short_factual.jsonl'
WORKLOAD_TYPES = ["short_factual", "multi_hop", "semantic_similarity"]


### Authenticate

Prompts for a Hugging Face access token (needs write access) if you're not already logged in. In Colab, this shows a login widget. Locally, run `huggingface-cli login` once beforehand instead, and this cell will just confirm you're authenticated.

In [3]:
from huggingface_hub import login, whoami

try:
    user_info = whoami()
    print(f"Already authenticated as: {user_info['name']}")
except Exception:
    print("Not authenticated yet, prompting for token...")
    login()
    user_info = whoami()
    print(f"Authenticated as: {user_info['name']}")

c:\project\empirical-rag-paradigm-benchmark\benchmarking-research\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Already authenticated as: Sakhiur


### Create the repo if it doesn't exist yet

Safe to run every time. If the repo already exists, this is a no-op (`exist_ok=True`). Only the first person to run this actually creates it; the second person's run just confirms it's already there.

In [4]:
from huggingface_hub import HfApi, create_repo

api = HfApi()

create_repo(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    private=PRIVATE,
    exist_ok=True,
)

print(f"Repo ready: https://huggingface.co/datasets/{REPO_ID}")

Repo ready: https://huggingface.co/datasets/Sakhiur/empirical-rag-paradigm-benchmark


### Discover QA pairs available locally

Looks for files shaped like `<category>_qa_pairs_<workload_type>.jsonl` directly under `data/question_answer_pairs/` (e.g. `cs.AI_qa_pairs_short_factual.jsonl`), for every combination of `ALL_CATEGORIES` x `WORKLOAD_TYPES`. Missing combinations are expected, not an error — only whatever's been generated so far gets uploaded.

In [5]:
found_qa_files = []  # list of (category, workload_type, local_path)

for workload_type in WORKLOAD_TYPES:
    for category in ALL_CATEGORIES:
        fname = f"{category}_qa_pairs_{workload_type}.jsonl"
        path = os.path.join(QA_PAIRS_DIR, fname)
        if os.path.exists(path):
            found_qa_files.append((category, workload_type, path))

print("QA pair files found locally:")
if found_qa_files:
    for category, workload_type, path in found_qa_files:
        n_lines = sum(1 for line in open(path, "r", encoding="utf-8") if line.strip())
        print(f"  {workload_type}/{category}: {n_lines} QA pairs ({path})")
else:
    print("  none found")

QA pair files found locally:
  short_factual/cs.AI: 198 QA pairs (..\data\question_answer_pairs\cs.AI_qa_pairs_short_factual.jsonl)


### Upload QA pair files

Uploaded under `question_answer_pairs/<workload_type>/<category>.jsonl` in the repo, e.g. `question_answer_pairs/short_factual/cs.AI.jsonl`. This loop is generic across `found_qa_files`, so it picks up whatever categories/workload types you've generated — today that's just short-factual/cs.AI, but multi-hop and semantic-similarity files will upload the same way once they exist locally.

In [6]:
for category, workload_type, local_path in found_qa_files:
    repo_path = f"question_answer_pairs/{workload_type}/{category}.jsonl"

    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=repo_path,
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        commit_message=f"Upload {workload_type} QA pairs for {category} (by {user_info['name']})",
    )
    print(f"Uploaded {repo_path}")

Uploaded question_answer_pairs/short_factual/cs.AI.jsonl


### Verify: list everything currently in the repo

Confirms what's actually on the Hub right now, combining both collaborators' uploads if both have run this. Good sanity check before either of you starts building on top of it (embeddings, clustering, QA generation), to make sure you're both seeing the same thing.

In [7]:
repo_files = api.list_repo_files(repo_id=REPO_ID, repo_type=REPO_TYPE)

abstracts_on_hub = sorted(f for f in repo_files if f.startswith("abstracts/"))
chunks_on_hub = sorted(f for f in repo_files if f.startswith("pdf_chunks/"))
artifacts_on_hub = sorted(f for f in repo_files if f.startswith("pipeline_artifacts/"))

print(f"Repo: https://huggingface.co/datasets/{REPO_ID}\n")

print(f"Abstract files on Hub ({len(abstracts_on_hub)}):")
for f in abstracts_on_hub:
    print(f"  {f}")

print(f"\nPDF chunk files on Hub ({len(chunks_on_hub)}):")
for f in chunks_on_hub:
    print(f"  {f}")

print(f"\nPipeline artifacts on Hub ({len(artifacts_on_hub)}):")
for f in artifacts_on_hub:
    print(f"  {f}")

qa_pairs_on_hub = sorted(f for f in repo_files if f.startswith("question_answer_pairs/"))
print(f"\nQA pair files on Hub ({len(qa_pairs_on_hub)}):")
for f in qa_pairs_on_hub:
    print(f"  {f}")

abstract_categories_on_hub = {f.split("/")[-1].replace(".jsonl", "") for f in abstracts_on_hub}
chunk_categories_on_hub = {f.split("/")[-1].replace(".jsonl", "") for f in chunks_on_hub}

missing_abstract_cats = set(ALL_CATEGORIES) - abstract_categories_on_hub
missing_chunk_cats = set(ALL_CATEGORIES) - chunk_categories_on_hub

print(f"\nStill missing from abstracts (across both collaborators): {sorted(missing_abstract_cats) if missing_abstract_cats else 'none, all 10 present'}")
print(f"Still missing from PDF chunks (across both collaborators): {sorted(missing_chunk_cats) if missing_chunk_cats else 'none, all 10 present'}")

for workload_type in WORKLOAD_TYPES:
    cats_on_hub_for_workload = {
        f.split("/")[-1].replace(".jsonl", "")
        for f in qa_pairs_on_hub
        if f.startswith(f"question_answer_pairs/{workload_type}/")
    }
    missing = set(ALL_CATEGORIES) - cats_on_hub_for_workload
    print(f"Still missing from {workload_type} QA pairs (across both collaborators): {sorted(missing) if missing else 'none, all 10 present'}")

Repo: https://huggingface.co/datasets/Sakhiur/empirical-rag-paradigm-benchmark

Abstract files on Hub (10):
  abstracts/by_category/cs.AI.jsonl
  abstracts/by_category/cs.CL.jsonl
  abstracts/by_category/cs.CR.jsonl
  abstracts/by_category/cs.CV.jsonl
  abstracts/by_category/cs.DB.jsonl
  abstracts/by_category/cs.DC.jsonl
  abstracts/by_category/cs.IR.jsonl
  abstracts/by_category/cs.LG.jsonl
  abstracts/by_category/cs.NE.jsonl
  abstracts/by_category/cs.SE.jsonl

PDF chunk files on Hub (10):
  pdf_chunks/cs.AI.jsonl
  pdf_chunks/cs.CL.jsonl
  pdf_chunks/cs.CR.jsonl
  pdf_chunks/cs.CV.jsonl
  pdf_chunks/cs.DB.jsonl
  pdf_chunks/cs.DC.jsonl
  pdf_chunks/cs.IR.jsonl
  pdf_chunks/cs.LG.jsonl
  pdf_chunks/cs.NE.jsonl
  pdf_chunks/cs.SE.jsonl

Pipeline artifacts on Hub (4):
  pipeline_artifacts/category_counts.csv
  pipeline_artifacts/chunk_hashes.json
  pipeline_artifacts/page_type_counts.csv
  pipeline_artifacts/routing_ledger.jsonl

QA pair files on Hub (1):
  question_answer_pairs/short

### How to load this back later (reference, not run here)

For the next notebook (embeddings, clustering, QA generation), pull files like this from either Colab or local. `hf_hub_download` caches locally, so repeated calls across sessions do not re-download unless the file changed on the Hub.

In [12]:
from huggingface_hub import hf_hub_download

example_path = hf_hub_download(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    filename="abstracts/by_category/cs.AI.jsonl",  # change to whatever category you need
)
print(f"Downloaded to: {example_path}")

with open(example_path, "r", encoding="utf-8") as f:
    first_line = f.readline()
print(f"First record preview: {first_line[:200]}...")

Downloaded to: C:\Users\Sakhiur\.cache\huggingface\hub\datasets--Sakhiur--empirical-rag-paradigm-benchmark\snapshots\8616e8837d5cb1ad97feac2b583b3e32fecbc449\abstracts\by_category\cs.AI.jsonl
First record preview: {"id": "2109.14155", "title": "Customs Fraud Detection in the Presence of Concept Drift", "abstract": "Capturing the changing trade pattern is critical in customs fraud detection. As new goods are imp...


c:\project\empirical-rag-paradigm-benchmark\benchmarking-research\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sakhiur\.cache\huggingface\hub\datasets--Sakhiur--empirical-rag-paradigm-benchmark. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
